# MAPK signalling as a Bigraphical Reactive System

_Investigation `mapk-signalling` — coder reproduction notebook._

**Question.** Can the MAPK signalling cycle be expressed as a Milner-style Bigraphical
Reactive System and reproduce, under Gillespie SSA, the spatial signalling
behaviour of the prior live demo — MEK·ERK complex formation and phospho-ERK
translocation into the nucleus?

A single-study demonstration that the Bigraphical Reactive System engine in
this workspace reproduces the MAPK signalling cycle from the prior live demo,
now runnable and inspectable inside the workbench.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/viva-reactive-system--workbench').is_dir():
    REPO = Path('/Users/eranagmon/code/viva-reactive-system--workbench')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_reactive_system.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: MAPK BRS basics (`mapk-brs-basics`)

**Question.** Run as a Bigraphical Reactive System under Gillespie SSA, does the MAPK
signalling cycle reproduce the qualitative behaviour of the prior live demo
— MEK·ERK complexes forming in the cytoplasm and phospho-ERK accumulating
in the nucleus by the end of the run?

**Objective.** Reproduce the prior MAPK BRS live demo inside the workbench: run the MAPK
signalling cycle as a Milner-style Bigraphical Reactive System under
Gillespie SSA and surface its basic systems — the initial nested-cell
bigraph, the seven redex→reactum rewrite rules, the per-compartment
substrate population time series, and the structural snapshots/animation of
phospho-ERK translocating into the nucleus.

**Claim.** Run as a Bigraphical Reactive System under Gillespie SSA, the MAPK cycle
reproduces the prior live demo: MEK·substrate complexes form in the
cytoplasm and phospho-ERK accumulates in the nucleus (0 → 144 over 120 ticks).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mapk_brs` | `pbg_reactive_system.composites.mapk_brs` | 120 | — |
| `seed-7` | `pbg_reactive_system.composites.mapk_brs` | 120 | seed=7 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_reactive_system.composites.mapk_brs`** — `spec_pbg_reactive_system_composites_mapk_brs` (a plain, editable dict)


_composite spec file for `pbg_reactive_system.composites.mapk_brs` not found under `pbg_reactive_system/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mapk-brs-basics ===
STUDY = 'mapk-brs-basics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_mapk_brs = 120
INTERVAL_mapk_brs = 0.1
STEPS_seed_7 = 120
INTERVAL_seed_7 = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_reactive_system_composites_mapk_brs, {'interval': INTERVAL_mapk_brs}, core=core)
        comp.run(STEPS_mapk_brs)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_reactive_system_composites_mapk_brs, {'interval': INTERVAL_seed_7}, core=core)
        comp.run(STEPS_seed_7)  # writes the composite's declared emitter
    print(f'ran 2 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**initial-bigraph**


In [ ]:
# initial-bigraph
show_viz(_render_one('image:charts/mapk_brs_viz.png', {}, RUNS_DB, STUDY_YAML))

**population-timeseries**


In [ ]:
# population-timeseries
show_viz(_render_one('image:charts/mapk_brs_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**structural-snapshots**


In [ ]:
# structural-snapshots
show_viz(_render_one('image:charts/mapk_brs_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**rule-catalog**


In [ ]:
# rule-catalog
show_viz(_render_one('image:charts/mapk_brs_trace.png', {}, RUNS_DB, STUDY_YAML))

**animation**


In [ ]:
# animation
show_viz(_render_one('image:charts/mapk_brs_animation.gif', {}, RUNS_DB, STUDY_YAML))

**species-timeseries**


In [ ]:
# species-timeseries
show_viz(_render_one('local:MapkSpeciesTimeseries', {'inputs_map': {'time': 'global_time'}}, RUNS_DB, STUDY_YAML))

**animation-interactive**


In [ ]:
# animation-interactive
show_viz(_render_one('local:MapkAnimation', {'inputs_map': {'time': 'global_time'}, 'total_time': 60, 'n_keyframes': 12, 'n_intermediate': 4}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| nucleus-accumulates-perk | kind=listener_final path=nucleus_pERK window=whole_run | op at_least low 1 provenance {'kind': 'theory', 'note': 'The translocate_perk_in rule moves pERK into the nucleus, so a run must end with nuclear pERK > 0.'} |
| nucleus-starts-empty | kind=listener_initial path=nucleus_pERK window=whole_run | op equals value 0 provenance {'kind': 'theory', 'note': 'initial_mapk_state seeds no substrate in the nucleus.'} |
| complexes-form | kind=listener_max path=complexes window=whole_run | op at_least low 1 provenance {'kind': 'theory', 'note': 'The phosphorylate rule binds MEK to a substrate', 'creating a link-graph bond.': None} |
| population-grows | kind=listener_final path=total_substrate window=whole_run | op at_least low 5 provenance {'kind': 'theory', 'note': "Starts at 4 copies; the demo's population time series grows well beyond that."} |
